# MASIVE-ALS — Docking en Google Colab con Vina-GPU (T4)

**Tanda NUEVA con Vina-GPU-2.1** (usa la tarjeta grafica, mas rapido que CPU).

**Como usar:**
1. Entorno de ejecucion -> Cambiar tipo de entorno -> **T4 GPU**.
2. Ejecutar las celdas en orden. El binario Vina-GPU y los datos se descargan
   solos desde el repo publico (GitHub `masive-als-data`), no hay que subir nada.

**Validacion incluida:** la celda 5 primero re-acopla una muestra de pares YA
hechos con Vina 1.2.5 (seed 42) y compara energias. Si difieren mucho, avisa.


In [ ]:
# CELDA 1: Verificar GPU (REQUISITO para Vina-GPU)
!nvidia-smi
print()
print('Vina-GPU NECESITA esta GPU para funcionar. Si no aparece, activa T4 GPU.')


In [ ]:
# CELDA 2: Instalar dependencias de Vina-GPU (Boost runtime 1.74)
!apt-get install -y libboost-program-options1.74.0 libboost-system1.74.0 libboost-filesystem1.74.0 2>&1 | tail -2
!ldconfig -p | grep -i boost | grep -E 'program_options|system|filesystem' | head -3
print('Dependencias listas')


In [ ]:
# CELDA 5: Docking con Vina-GPU-2.1 en Colab (T4 GPU)
import csv, glob, os, shutil, subprocess, tarfile, time, urllib.request

WORK = '/content/masive_als'
BIN_DIR = '/content/vinagpu_linux'
VINA_GPU_URL = 'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/vinagpu_linux.tar.gz'
SUBCONJUNTO = 'impares'   # Colab hace impares, Kaggle pares
VALIDAR = False           # False = tanda completa (validacion YA hecha 12 pares, OK)
THREAD = 8000
EN_KAGGLE = False         # solo Colab
IN = None

# --- 0) Verificar GPU (REQUISITO: Vina-GPU sin GPU no funciona) ---
print('=== VERIFICACION GPU ===', flush=True)
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=30)
print('nvidia-smi rc=%d' % gpu.returncode, flush=True)
if gpu.returncode != 0:
    print('ERROR: Vina-GPU necesita GPU NVIDIA. Activa T4 GPU.', flush=True)
    print('En Colab: Entorno de ejecucion > Cambiar tipo > T4 GPU.', flush=True)
    raise SystemExit(1)
print([l for l in gpu.stdout.splitlines() if 'Tesla' in l or 'NVIDIA' in l or 'P100' in l or 'T4' in l][:2], flush=True)

# --- 1) Obtener binario Vina-GPU (desde GitHub) ---
def buscar_binario(base):
    if not base or not os.path.exists(base):
        return None
    hits = glob.glob(base + '/**/AutoDock-Vina-GPU-2-1', recursive=True)
    return hits[0] if hits else None

VINA_GPU_BIN = buscar_binario(BIN_DIR)
if not VINA_GPU_BIN:
    os.makedirs(BIN_DIR, exist_ok=True)
    pkg = '/content/vinagpu_linux.tar.gz'
    if not os.path.exists(pkg):
        urllib.request.urlretrieve(VINA_GPU_URL, pkg)
    with tarfile.open(pkg) as t:
        t.extractall(BIN_DIR)
    VINA_GPU_BIN = buscar_binario(BIN_DIR)
if not VINA_GPU_BIN:
    print('ERROR: no se pudo obtener el binario Vina-GPU.', flush=True)
    raise SystemExit(1)
BIN_DIR = os.path.dirname(VINA_GPU_BIN)
os.chmod(VINA_GPU_BIN, 0o755)
print('Binario Vina-GPU listo:', VINA_GPU_BIN, flush=True)
print('Kernels:', sorted(glob.glob(BIN_DIR + '/OpenCL/src/kernels/*.cl')), flush=True)

# --- 2) Datos (descarga desde GitHub) ---
for sub in ['receptores', 'ligandos', 'resultados']:
    os.makedirs(WORK + '/' + sub, exist_ok=True)

def es_receptor(n):
    return n in ('TDP43.pdbqt', 'SOD1.pdbqt', 'FUS.pdbqt')

URLS = [
    'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/colab_receptores_plano.tar.gz',
    'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/colab_ligandos50_plano.tar.gz',
    'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/ligandos_ela_plano.tar.gz',
]
extract_dir = '/content/tmp_extract'
if os.path.exists(extract_dir):
    shutil.rmtree(extract_dir)
os.makedirs(extract_dir)
for url in URLS:
    dest = '/content/' + os.path.basename(url)
    if os.path.exists(dest):
        os.remove(dest)
    urllib.request.urlretrieve(url, dest)
for tar in sorted(glob.glob('/content/*.tar.gz')):
    if 'vinagpu' in tar:
        continue
    with tarfile.open(tar) as t:
        try:
            t.extractall(extract_dir, filter='data')
        except TypeError:
            t.extractall(extract_dir)
for raiz, _, archivos in os.walk(extract_dir):
    for a in archivos:
        if a.endswith('.pdbqt'):
            destino = WORK + '/receptores/' if es_receptor(a) else WORK + '/ligandos/'
            shutil.copy(os.path.join(raiz, a), destino + a)

# --- 2b) PDBQT reparados (desde GitHub, sobreescriben los corruptos) ---
REPARADOS = ['CHEMBL1076399', 'CHEMBL1163427', 'CHEMBL1203109', 'CHEMBL1203132',
             'CHEMBL1203140', 'CHEMBL1203155', 'CHEMBL1203199', 'CHEMBL1203224',
             'CHEMBL1203252', 'CHEMBL1204421', 'CHEMBL1207772', 'CHEMBL1208195',
             'CHEMBL152893']
n_cop = 0
for r in REPARADOS:
    url = 'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/ligandos_reparados/' + r + '.pdbqt'
    try:
        urllib.request.urlretrieve(url, WORK + '/ligandos/' + r + '.pdbqt')
        n_cop += 1
    except Exception as ex:
        print('AVISO reparado', r, str(ex)[:60], flush=True)
print('PDBQT reparados copiados:', n_cop, flush=True)
print('Receptores:', len(glob.glob(WORK + '/receptores/*.pdbqt')),
      '| Ligandos:', len(glob.glob(WORK + '/ligandos/*.pdbqt')), flush=True)

# --- 3) Receptores ---
RECEPTORES = {
    'TDP43': {'archivo': WORK + '/receptores/TDP43.pdbqt', 'centro': [28.3, 43.7, 52.5], 'tamano': [25, 25, 25]},
    'SOD1': {'archivo': WORK + '/receptores/SOD1.pdbqt', 'centro': [27.9, 111.8, 64.4], 'tamano': [25, 25, 25]},
    'FUS': {'archivo': WORK + '/receptores/FUS.pdbqt', 'centro': [-14.5, 15.1, -7.8], 'tamano': [25, 25, 25]},
}

# --- 4) FUS plano (sin MODEL/ENDMDL) ---
fus = RECEPTORES['FUS']['archivo']
try:
    lines = open(fus).read().splitlines()
    ini = next((i for i, l in enumerate(lines) if l.startswith('MODEL')), None)
    fin = next((i for i, l in enumerate(lines) if l.strip() == 'ENDMDL'), None)
    keep = lines[ini + 1:fin] if (ini is not None and fin is not None and fin >= ini) else lines
    if not any(l.startswith(('ATOM', 'HETATM')) for l in keep):
        keep = lines
    open(fus, 'w').write('\n'.join(keep) + '\n')
    print('FUS plano: ATOM=%d' % sum(1 for l in keep if l.startswith(('ATOM', 'HETATM'))), flush=True)
except Exception as ex:
    print('ERROR reparando FUS:', str(ex)[:100], flush=True)

# --- 5) CSV (reanuda) ---
CSV = WORK + '/resultados/resultados_vinagpu.csv'
if not os.path.exists(CSV):
    with open(CSV, 'w', newline='') as f:
        csv.writer(f).writerow(['ligand', 'target', 'energy', 'timestamp'])
SALTADOS = WORK + '/resultados/saltados_vinagpu.txt'
saltados = set()
if os.path.exists(SALTADOS):
    saltados = set(l.strip() for l in open(SALTADOS) if l.strip())
hechos = {}
for r in csv.DictReader(open(CSV)):
    hechos.setdefault(r['ligand'], set()).add(r['target'])

# --- 6) Base Colab (pares ya hechos) ---
BASE = WORK + '/resultados/resultados_colab.csv'
try:
    if EN_KAGGLE:
        shutil.copy(IN + '/resultados_colab.csv', BASE)
    else:
        urllib.request.urlretrieve(
            'https://raw.githubusercontent.com/fredy30-Rojas/masive-als-data/main/resultados_colab.csv', BASE)
    for r in csv.DictReader(open(BASE)):
        hechos.setdefault(r['ligand'], set()).add(r['target'])
    print('Base Colab fusionada: %d pares ya hechos' % sum(len(v) for v in hechos.values()), flush=True)
except Exception as ex:
    print('AVISO sin base Colab:', str(ex)[:80], flush=True)

# --- 7) Seleccion de pares ---
ligs = sorted(glob.glob(WORK + '/ligandos/*.pdbqt'))
if SUBCONJUNTO in ('pares', 'impares'):
    paridad = 0 if SUBCONJUNTO == 'pares' else 1
    ligs = [l for i, l in enumerate(ligs) if i % 2 == paridad]
print('Ligandos (subconjunto=%s):' % SUBCONJUNTO, len(ligs), flush=True)

pendientes = []
for lig in ligs:
    nombre = os.path.basename(lig).replace('.pdbqt', '')
    if nombre in saltados:
        continue
    para = hechos.get(nombre, set())
    for target in RECEPTORES:
        if target not in para:
            pendientes.append((lig, nombre, target))
print('Pares pendientes:', len(pendientes), flush=True)

if VALIDAR and pendientes:
    completos = {n for n in hechos if len(hechos[n]) >= 3}
    muestras = [p for p in pendientes if p[1] in completos][:30]
    if muestras:
        print('VALIDACION: re-acoplo %d pares ya hechos' % len(muestras), flush=True)
        pendientes = muestras

# --- 8) Acoplar con Vina-GPU (config file por par) ---
# NOTA: Vina-GPU-2.1 NO acepta --exhaustiveness ni --cpu (comentados en su parser).
# El control de esfuerzo es --thread (numero de tareas de computo en la GPU).
# Ejecutamos con cwd=BIN_DIR para que encuentre los kernels ./OpenCL/ (default_work_path=".").
def acoplar(lig, target, thread=8000):
    info = RECEPTORES[target]
    cfg = '/tmp/cfg_%s_%s.txt' % (os.path.basename(lig).replace('.pdbqt', '')[:20], target)
    with open(cfg, 'w') as f:
        f.write('receptor = %s\n' % info['archivo'])
        f.write('ligand = %s\n' % lig)
        f.write('center_x = %s\n' % info['centro'][0])
        f.write('center_y = %s\n' % info['centro'][1])
        f.write('center_z = %s\n' % info['centro'][2])
        f.write('size_x = %s\n' % info['tamano'][0])
        f.write('size_y = %s\n' % info['tamano'][1])
        f.write('size_z = %s\n' % info['tamano'][2])
        f.write('num_modes = 3\n')
        f.write('seed = 42\n')
        f.write('thread = %d\n' % thread)
    try:
        r = subprocess.run([VINA_GPU_BIN, '--config', cfg], capture_output=True, text=True,
                           timeout=1800, cwd=BIN_DIR)
        out = (r.stdout or '') + (r.stderr or '')
        if r.returncode != 0:
            return None, 'vinagpu rc=%d %s' % (r.returncode, out[-200:])
        for ln in out.splitlines():
            s = ln.split()
            if len(s) >= 2 and s[0] == '1':
                try:
                    return round(float(s[1]), 4), None
                except ValueError:
                    pass
        return None, 'sin afinidad: ' + out[-150:]
    except Exception as ex:
        return None, str(ex)[:100]

def tiene_atomos(lig):
    try:
        with open(lig, errors='replace') as f:
            return any(l.startswith(('ATOM', 'HETATM')) for l in f)
    except Exception:
        return False

if pendientes:
    t0 = time.time()
    n_ok = 0
    for lig, nombre, target in pendientes:
        if not tiene_atomos(lig):
            print('LIGANDO_VACIO', nombre, '- se salta', flush=True)
            with open(SALTADOS, 'a') as f:
                f.write(nombre + '\n')
            continue
        energia, err = acoplar(lig, target)
        if energia is not None:
            with open(CSV, 'a', newline='') as f:
                csv.writer(f).writerow([nombre, target, energia, time.strftime('%Y-%m-%d %H:%M:%S')])
            n_ok += 1
        else:
            print('ERROR', nombre, target, err, flush=True)
        if n_ok % 5 == 0 and n_ok > 0:
            print('[%d acoplados] %.1f min' % (n_ok, (time.time() - t0) / 60), flush=True)
    print('Acoplados ahora:', n_ok, flush=True)

# --- 9) Comparacion con Vina 1.2.5 ---
if VALIDAR and os.path.exists(BASE):
    print()
    print('=== COMPARACION Vina-GPU vs Vina 1.2.5 (seed 42) ===', flush=True)
    refs = {}
    for r in csv.DictReader(open(BASE)):
        refs.setdefault(r['ligand'], {})[r['target']] = float(r['energy'])
    diffs = []
    for r in csv.DictReader(open(CSV)):
        q = float(r['energy'])
        ref = refs.get(r['ligand'], {}).get(r['target'])
        if ref is not None:
            diffs.append((r['ligand'], r['target'], ref, q, abs(q - ref)))
    if diffs:
        for lig, tgt, ref, q, d in sorted(diffs, key=lambda x: -x[4])[:15]:
            print('%-22s %-5s Vina=%7.2f GPU=%7.2f diff=%5.2f' % (lig, tgt, ref, q, d), flush=True)
        mean = sum(d[4] for d in diffs) / len(diffs)
        maxd = max(d[4] for d in diffs)
        print('MEDIA diff=%.3f | MAX diff=%.3f | n=%d' % (mean, maxd, len(diffs)), flush=True)
        if maxd > 1.5:
            print('ALERTA: diferencias grandes - NO mezclar motores sin revisar.', flush=True)
        else:
            print('OK: diferencias pequenas, motores compatibles.', flush=True)

print(flush=True)
print('=== TANDAS COMPLETADAS ===', flush=True)
print('Filas en CSV:', len(list(csv.DictReader(open(CSV)))), flush=True)


In [ ]:
# CELDA 6: Resumen de resultados (con respaldo completo en la celda)
import csv, os
rows = list(csv.DictReader(open(CSV)))
print('Total resultados:', len(rows))
if rows:
    best = sorted(rows, key=lambda x: float(x['energy']))[:10]
    print()
    print('Top 10:')
    for r in best:
        print('  ', r['ligand'], r['target'], r['energy'])

print()
print('Descargar: panel izquierdo -> masive_als/resultados/resultados_vinagpu.csv')
print()
print('RESULTADOS TOTALES (respaldo en celda):')
print(open(CSV).read())
